In [1]:
!git clone https://github.com/OdincovMD/img2txt.git

Cloning into 'img2txt'...
remote: Enumerating objects: 643, done.
remote: Counting objects: 100% (113/113), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 643 (delta 30), reused 91 (delta 14), pack-reused 530 (from 1)
Receiving objects: 100% (643/643), 12.30 MiB | 22.53 MiB/s, done.
Resolving deltas: 100% (277/277), done.


In [2]:
import sys
from pathlib import Path

img2txt_path = Path("/kaggle/working/img2txt")
sys.path.insert(0, str(img2txt_path))

In [3]:
!pip install python-resize-image -q
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.9 MB/s eta 0:00:00a 0:00:01


In [4]:
import pandas as pd
from extraction.feature_extraction_batch import extract_features_batch, images_to_df
from analysis.distribution_analysis import main as analysis
from bucketing.feature_bucketing_batch import bucket_features_batch
from model.run import xgb
from model.mlp_optuna import mlp
from model.mlp_run import mlp_run


# Пути
IMAGE_DIR = "/kaggle/input/datasets/mihailodin1/all-image-skin"
YOLO_WEIGHTS = "/kaggle/input/datasets/mihailodin1/skinweightalllast-version/weight/mask_builder_yolo.pt"
UNET_WEIGHTS = "/kaggle/input/datasets/mihailodin1/skinweightalllast-version/weight/mask_builder_unet.pth"
FEATURES_CSV = "/kaggle/working/img2txt/bucketed_features.csv"
ANNOTATIONS_CSV = "/kaggle/working/img2txt/annotations.csv"

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:
from extraction.segmentation import  main

In [9]:
from PIL import Image
import numpy as np

mask = main(
    "/kaggle/input/datasets/mihailodin1/all-image-skin/100.jpg",
    YOLO_WEIGHTS, 
    UNET_WEIGHTS
)

mask_img = Image.fromarray(mask)
mask_img.save("100_mask.jpg")

## Шаг 1. Извлечение описательных статистик.

### 1.1 Разметка набора изображений.

In [5]:
# df = images_to_df(IMAGE_DIR)
# df = extract_features_batch(df, yolo_weights=YOLO_WEIGHTS, unet_weights=UNET_WEIGHTS)
# df.to_csv(FEATURES_CSV, index=False)
# print(f"Извлечено: {len(df)} изображений, успешно: {(df['status'] == 'success').sum()}")

### 1.2 Загрузка готовой разметки.

In [6]:
df = pd.read_csv(FEATURES_CSV)

In [7]:
df.head(2)

,image_path,status,mean_H_lesion,mean_S_lesion,mean_V_lesion,std_H_lesion,std_S_lesion,std_V_lesion,color_balance_B,color_balance_G,...,bucket_lbp_entropy,bucket_lbp_mean,bucket_lbp_std,bucket_lbp_median,bucket_percent_dark_pixels,bucket_percent_white_pixels,bucket_percent_red_pixels,bucket_percent_blue_pixels,bucket_percent_outlier_bright_pixels,bucket_percent_outlier_dark_pixels
0,images/100.jpg,success,8.844754,143.157422,127.225247,3.760556,57.882412,42.754618,0.235202,0.304292,...,энтропия_lbp:средняя,среднее_lbp:высокое,разброс_lbp:средний,медиана_lbp:типичная,темные_пиксели:нет,белые_пиксели:единичные,красные_пиксели:много,синие_пиксели:нет,яркие_выбросы:нет,темные_выбросы:нет
1,images/1000.jpg,success,9.441791,130.862138,202.150413,2.038567,23.189567,14.328646,0.229396,0.303959,...,энтропия_lbp:высокая,среднее_lbp:среднее,разброс_lbp:высокий,медиана_lbp:типичная,темные_пиксели:нет,белые_пиксели:нет,красные_пиксели:преобладают,синие_пиксели:нет,яркие_выбросы:много,темные_выбросы:умеренно


## Шаг 2. Бакетирование


### 2.1 Анализ распределений и подбор параметров.

In [8]:
# sys.argv = [
#     "/kaggle/working/img2txt/analysis/distribution_analysis.py"
#     "--csv", FEATURES_CSV
# ]
# analysis()

### 2.2 Бакетирование признаков

In [9]:
df = bucket_features_batch(df)

100%|██████████| 4137/4137 [00:00<00:00, 12792.13it/s]


## Шаг 3. Ранжирование (Выбор топ-10 важных признаков XGBoost)

### 3.1 Обучение модели

#### 3.1.1 XGB

In [10]:
optuna_kwargs = {
    "learning_rate": {"type": "float", "low": 0.003, "high": 0.08, "log": True},
    "n_estimators": {"type": "int", "low": 500, "high": 1600},
    "reg_lambda": {"type": "float", "low": 0.5, "high": 6.0},
    "reg_alpha": {"type": "float", "low": 1e-4, "high": 0.3, "log": True},
    "subsample": {"type": "float", "low": 0.6, "high": 1.0},
    "colsample_bytree": {"type": "float", "low": 0.55, "high": 1.0},
    "min_child_weight": {"type": "int", "low": 1, "high": 8},
    "gamma": {"type": "float", "low": 0.0, "high": 5.0},
    "max_bin": {"type": "categorical", "choices": [128, 256, 512]},
    "max_depth": {"type": "int", "low": 4, "high": 7},
}

result = xgb(
    features_csv=FEATURES_CSV,
    annotations_csv=ANNOTATIONS_CSV,
    feature_set="selected_buckets",
    model_type="xgb",
    n_trials=30,
    checkpoint_path="/kaggle/working/xgb_importance.pkl",
    calibrate_label_bias=True,
    kwargs=optuna_kwargs,
)

  XGBoost importance model


[I 2026-04-22 13:33:01,334] A new study created in memory with name: no-name-04cdde06-4a31-4f49-819d-cc9ef61be7a0
[I 2026-04-22 13:33:10,702] Trial 0 finished with value: 0.6158373015873017 and parameters: {'learning_rate': 0.02081753027680346, 'n_estimators': 913, 'reg_lambda': 2.2516076734650277, 'reg_alpha': 0.00034788364089838824, 'subsample': 0.7511234688764071, 'colsample_bytree': 0.9449196745542713, 'min_child_weight': 5, 'gamma': 0.490883090300579, 'max_bin': 128, 'max_depth': 7}. Best is trial 0 with value: 0.6158373015873017.
[I 2026-04-22 13:33:25,955] Trial 1 finished with value: 0.6158809523809523 and parameters: {'learning_rate': 0.007012023114739468, 'n_estimators': 813, 'reg_lambda': 3.656928588575444, 'reg_alpha': 0.016886952768850588, 'subsample': 0.7756296926165571, 'colsample_bytree': 0.7632434627572626, 'min_child_weight': 7, 'gamma': 3.0830733698150077, 'max_bin': 128, 'max_depth': 7}. Best is trial 1 with value: 0.6158809523809523.
[I 2026-04-22 13:33:38,707] Tri


  BEST PARAMS
{'learning_rate': 0.012003521459161039, 'n_estimators': 1068, 'reg_lambda': 3.838581905814251, 'reg_alpha': 0.0007207860441757997, 'subsample': 0.7955384667275734, 'colsample_bytree': 0.9221258222564838, 'min_child_weight': 3, 'gamma': 0.006110552386934809, 'max_bin': 512, 'max_depth': 5}
Best score: 0.6300
Final score: 0.6300
Calibration: alpha=0.325, score 0.6300 -> 0.6355, precision 0.5910 -> 0.5960, recall 0.6690 -> 0.6750
Targeted calibration labels: palette, borders, delta_V_top_bottom, delta_V_left_right, delta_V_center_periphery, structure_order, delta_H_center_periphery, elongation

Most under-selected labels after calibration:
  palette: true=0.027, selected=0.000, delta=-0.027, recall@10=0.000
  delta_V_top_bottom: true=0.032, selected=0.007, delta=-0.025, recall@10=0.143
  delta_H_center_periphery: true=0.033, selected=0.012, delta=-0.021, recall@10=0.207
  delta_V_left_right: true=0.016, selected=0.000, delta=-0.016, recall@10=0.000
  perimeter: true=0.050, 

In [11]:
result = xgb(
    features_csv=FEATURES_CSV,
    annotations_csv=ANNOTATIONS_CSV,
    feature_set="numeric_only",
    model_type="xgb",
    n_trials=30,
    checkpoint_path="/kaggle/working/xgb_numeric.pkl",
    calibrate_label_bias=True,
    kwargs=optuna_kwargs,
)

  XGBoost importance model


[I 2026-04-22 13:38:59,251] A new study created in memory with name: no-name-af313757-6007-4a5b-b5ae-c84516a0ed82
[I 2026-04-22 13:39:05,134] Trial 0 finished with value: 0.6223035714285714 and parameters: {'learning_rate': 0.007864755004903332, 'n_estimators': 1238, 'reg_lambda': 2.159451704666281, 'reg_alpha': 0.0030667667256469924, 'subsample': 0.9242795359568242, 'colsample_bytree': 0.9904013490787407, 'min_child_weight': 6, 'gamma': 4.276337474424644, 'max_bin': 512, 'max_depth': 7}. Best is trial 0 with value: 0.6223035714285714.
[I 2026-04-22 13:39:09,744] Trial 1 finished with value: 0.6229007936507938 and parameters: {'learning_rate': 0.033192148861190686, 'n_estimators': 508, 'reg_lambda': 4.738257648318553, 'reg_alpha': 0.23403549433039092, 'subsample': 0.6445415985580548, 'colsample_bytree': 0.7852969255905533, 'min_child_weight': 5, 'gamma': 0.4002029558476955, 'max_bin': 512, 'max_depth': 6}. Best is trial 1 with value: 0.6229007936507938.
[I 2026-04-22 13:39:14,943] Tria


  BEST PARAMS
{'learning_rate': 0.02344904562091069, 'n_estimators': 1482, 'reg_lambda': 3.0891680259237186, 'reg_alpha': 0.00020692195160681276, 'subsample': 0.8083782144522911, 'colsample_bytree': 0.8653303821810976, 'min_child_weight': 8, 'gamma': 0.9137946563751154, 'max_bin': 512, 'max_depth': 5}
Best score: 0.6246
Final score: 0.6246
Calibration: alpha=0.275, score 0.6246 -> 0.6325, precision 0.5860 -> 0.5930, recall 0.6631 -> 0.6720
Targeted calibration labels: palette, borders, delta_V_top_bottom, delta_V_left_right, delta_V_center_periphery, structure_order, delta_H_center_periphery, elongation

Most under-selected labels after calibration:
  delta_V_top_bottom: true=0.032, selected=0.004, delta=-0.028, recall@10=0.036
  palette: true=0.027, selected=0.000, delta=-0.027, recall@10=0.000
  borders: true=0.031, selected=0.008, delta=-0.023, recall@10=0.148
  perimeter: true=0.050, selected=0.033, delta=-0.017, recall@10=0.477
  delta_V_left_right: true=0.016, selected=0.002, de

In [12]:
result = xgb(
    features_csv=FEATURES_CSV,
    annotations_csv=ANNOTATIONS_CSV,
    feature_set="all_buckets",
    model_type="xgb",
    n_trials=30,
    checkpoint_path="/kaggle/working/xgb_buckets.pkl",
    calibrate_label_bias=True,
    kwargs=optuna_kwargs,
)

  XGBoost importance model


[I 2026-04-22 13:41:42,005] A new study created in memory with name: no-name-af608e95-36e8-44fe-a212-ef259a08638e
[I 2026-04-22 13:41:58,372] Trial 0 finished with value: 0.6133035714285715 and parameters: {'learning_rate': 0.018785147532894772, 'n_estimators': 1272, 'reg_lambda': 3.994685707423318, 'reg_alpha': 0.08993178131678815, 'subsample': 0.6836804546190207, 'colsample_bytree': 0.7724064194604439, 'min_child_weight': 2, 'gamma': 4.6745706729919725, 'max_bin': 512, 'max_depth': 6}. Best is trial 0 with value: 0.6133035714285715.
[I 2026-04-22 13:42:14,732] Trial 1 finished with value: 0.6178035714285715 and parameters: {'learning_rate': 0.019292869832649807, 'n_estimators': 1141, 'reg_lambda': 2.8434381986942583, 'reg_alpha': 0.0002485249827307118, 'subsample': 0.6487960192946581, 'colsample_bytree': 0.6256453988504938, 'min_child_weight': 2, 'gamma': 2.4630849922184774, 'max_bin': 512, 'max_depth': 7}. Best is trial 1 with value: 0.6178035714285715.
[I 2026-04-22 13:42:48,419] T


  BEST PARAMS
{'learning_rate': 0.009196470420278192, 'n_estimators': 1162, 'reg_lambda': 0.6316481907935888, 'reg_alpha': 0.0058620054678203415, 'subsample': 0.8579198832089692, 'colsample_bytree': 0.8337466109008047, 'min_child_weight': 7, 'gamma': 0.5015544472158925, 'max_bin': 256, 'max_depth': 5}
Best score: 0.6238
Final score: 0.6238
Calibration: alpha=0.525, score 0.6238 -> 0.6339, precision 0.5860 -> 0.5950, recall 0.6617 -> 0.6729
Targeted calibration labels: palette, borders, delta_V_top_bottom, delta_V_left_right, delta_V_center_periphery, structure_order, delta_H_center_periphery, elongation

Most under-selected labels after calibration:
  palette: true=0.027, selected=0.001, delta=-0.026, recall@10=0.042
  perimeter: true=0.050, selected=0.028, delta=-0.022, recall@10=0.409
  delta_S_center_periphery: true=0.035, selected=0.019, delta=-0.016, recall@10=0.161
  delta_V_top_bottom: true=0.032, selected=0.016, delta=-0.016, recall@10=0.250
  structure_order: true=0.017, sele

In [13]:
result = xgb(
    features_csv=FEATURES_CSV,
    annotations_csv=ANNOTATIONS_CSV,
    feature_set="selected_buckets",
    model_type="xgb_classifier_chain",
    n_trials=30,
    checkpoint_path="/kaggle/working/xgb_classifier_chain_importance.pkl",
    calibrate_label_bias=True,
    kwargs=optuna_kwargs,
)

  XGBoost importance model


[I 2026-04-22 13:52:50,145] A new study created in memory with name: no-name-12707c58-496e-4e00-84d4-895ff8689784
[I 2026-04-22 13:54:20,607] Trial 0 finished with value: 0.6169742063492063 and parameters: {'learning_rate': 0.006584767165456215, 'n_estimators': 629, 'reg_lambda': 1.401929929758677, 'reg_alpha': 0.010342100318265974, 'subsample': 0.782980120611096, 'colsample_bytree': 0.6313733115240806, 'min_child_weight': 5, 'gamma': 1.1886220250483637, 'max_bin': 512, 'max_depth': 6}. Best is trial 0 with value: 0.6169742063492063.
[I 2026-04-22 13:55:26,428] Trial 1 finished with value: 0.6121567460317461 and parameters: {'learning_rate': 0.012879988511717106, 'n_estimators': 740, 'reg_lambda': 2.4970643122197327, 'reg_alpha': 0.002709090327588364, 'subsample': 0.6210078431537303, 'colsample_bytree': 0.7007418197701788, 'min_child_weight': 7, 'gamma': 4.48955634449045, 'max_bin': 128, 'max_depth': 4}. Best is trial 0 with value: 0.6169742063492063.
[I 2026-04-22 13:57:04,387] Trial 


  BEST PARAMS
{'learning_rate': 0.04152049537757619, 'n_estimators': 1145, 'reg_lambda': 3.1341914913934685, 'reg_alpha': 0.00030889519735979436, 'subsample': 0.7681268563374717, 'colsample_bytree': 0.7437417659296764, 'min_child_weight': 5, 'gamma': 0.06493981910207447, 'max_bin': 128, 'max_depth': 4}
Best score: 0.6206
Final score: 0.6206
Calibration: alpha=0.250, score 0.6206 -> 0.6230, precision 0.5820 -> 0.5840, recall 0.6591 -> 0.6621
Targeted calibration labels: palette, borders, delta_V_top_bottom, delta_V_left_right, delta_V_center_periphery, structure_order, delta_H_center_periphery, elongation

Most under-selected labels after calibration:
  delta_V_top_bottom: true=0.032, selected=0.009, delta=-0.023, recall@10=0.143
  palette: true=0.027, selected=0.005, delta=-0.022, recall@10=0.042
  perimeter: true=0.050, selected=0.032, delta=-0.018, recall@10=0.386
  borders: true=0.031, selected=0.014, delta=-0.017, recall@10=0.296
  elongation: true=0.023, selected=0.010, delta=-0.

#### 3.1.2 MLP

In [16]:
mlp_kwargs = {
    "batch_size": {"type": "categorical", "choices": [16, 32, 64]},
    "learning_rate": {"type": "float", "low": 1e-4, "high": 3e-3, "log": True},
    "weight_decay": {"type": "float", "low": 1e-6, "high": 5e-3, "log": True},
    "dropout": {"type": "float", "low": 0.1, "high": 0.5},
    "max_epochs": {"type": "int", "low": 40, "high": 100},
    "patience": {"type": "int", "low": 8, "high": 20},
    "hidden1": {"type": "categorical", "choices": [64, 128, 192, 256]},
    "hidden2": {"type": "categorical", "choices": [32, 64, 96, 128]},
}

result = mlp(
    features_csv=FEATURES_CSV,
    annotations_csv=ANNOTATIONS_CSV,
    feature_set="numeric_only",
    n_trials=50,
    n_splits=5,
    checkpoint_path="/kaggle/working/mlp_importance.pt",
    kwargs=mlp_kwargs,
)

[I 2026-04-22 14:22:08,478] A new study created in memory with name: no-name-cfc7d398-2b72-4003-849e-96e81a95ac08
[I 2026-04-22 14:22:18,729] Trial 0 finished with value: 0.5115813492063491 and parameters: {'batch_size': 32, 'learning_rate': 0.001253829250364845, 'weight_decay': 6.069655108855242e-06, 'dropout': 0.329282646189211, 'max_epochs': 83, 'patience': 11, 'hidden1': 64, 'hidden2': 64}. Best is trial 0 with value: 0.5115813492063491.
[I 2026-04-22 14:22:36,735] Trial 1 finished with value: 0.5096039682539683 and parameters: {'batch_size': 16, 'learning_rate': 0.0012933901465471973, 'weight_decay': 0.0036656639812927376, 'dropout': 0.3603767576014817, 'max_epochs': 54, 'patience': 18, 'hidden1': 192, 'hidden2': 96}. Best is trial 0 with value: 0.5115813492063491.
[I 2026-04-22 14:22:50,321] Trial 2 finished with value: 0.5041626984126983 and parameters: {'batch_size': 16, 'learning_rate': 0.00025462224819268817, 'weight_decay': 0.00027124626265539033, 'dropout': 0.13046542405624

Best score: 0.5325
Best params: {'batch_size': 16, 'learning_rate': 0.001940459759750493, 'weight_decay': 1.0059662006969984e-06, 'dropout': 0.13281780165796125, 'max_epochs': 65, 'patience': 16, 'hidden1': 128, 'hidden2': 128}

Top trials:
 number    value             datetime_start          datetime_complete               duration  params_batch_size  params_dropout  params_hidden1  params_hidden2  params_learning_rate  params_max_epochs  params_patience  params_weight_decay    state
     36 0.532497 2026-04-22 14:27:56.886607 2026-04-22 14:28:11.058150 0 days 00:00:14.171543                 16        0.132818             128             128              0.001940                 65               16             0.000001 COMPLETE
     42 0.530453 2026-04-22 14:28:59.855340 2026-04-22 14:29:12.147163 0 days 00:00:12.291823                 16        0.140844             128             128              0.002689                 70               13             0.000005 COMPLETE
     28 0.52

In [17]:
result = mlp_run(
    features_csv=FEATURES_CSV,
    annotations_csv=ANNOTATIONS_CSV,
    checkpoint_path="/kaggle/working/mlp_importance.pt",
    feature_set="numeric_only",
    skip_cv=True,
    max_epochs=100,
    kwargs={
        "batch_size": 32,
        "patience": 15,
        "learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "dropout": 0.3,
        "hidden_dims_override": (128, 64),
    },
)


  MLP importance model

Best validation score: 0.5015
Checkpoint saved to: /kaggle/working/mlp_importance.pt


In [23]:
df = pd.read_csv(FEATURES_CSV)
result = rank_features_batch_mlp(
    df.head(10),
    importance_model_path="/kaggle/working/mlp_importance.pt",
    k=10,
    verbose=True,
)

result[["image_path", "important_labels"]].head()

In [13]:
df = pd.read_csv(FEATURES_CSV)
result = rank_features_batch(
    df,
    importance_model_path="/kaggle/working/xgb_importance.pkl",
    k=10,
    verbose=True,
)

result[["image_path", "important_labels"]].head()

100%|██████████| 4137/4137 [00:00<00:00, 8302.06it/s]


,image_path,important_labels
0,images/100.jpg,"[shape:округлая, color_distance_euclidean:силь..."
1,images/1000.jpg,"[shape:овальная, color_distance_euclidean:умер..."
2,images/1001.jpg,"[shape:овальная, color_distance_euclidean:умер..."
3,images/1002.jpg,"[shape:овальная, color_distance_euclidean:выра..."
4,images/1003.jpg,"[shape:овальная, asymmetry:умеренная, color_di..."


In [11]:
from model.model import load_checkpoint
import pandas as pd

checkpoint = load_checkpoint("/kaggle/working/xgb_importance.pkl")

report = pd.DataFrame(checkpoint["calibration_metadata"]["calibrated_report"])
report.sort_values("share_delta")

,label,positives,selected,hits,true_share,selected_share,share_delta,recall_at_k,precision_when_selected
20,delta_V_top_bottom,28,4,2,0.030973,0.004,-0.026973,0.071429,0.500000
3,palette,24,0,0,0.026549,0.000,-0.026549,0.000000,0.000000
1,borders,27,4,1,0.029867,0.004,-0.025867,0.037037,0.250000
15,delta_H_center_periphery,29,14,7,0.032080,0.014,-0.018080,0.241379,0.500000
19,delta_V_left_right,14,0,0,0.015487,0.000,-0.015487,0.000000,0.000000
17,delta_V_center_periphery,18,5,2,0.019912,0.005,-0.014912,0.111111,0.400000
23,glcm_energy,13,0,0,0.014381,0.000,-0.014381,0.000000,0.000000
11,perimeter,44,36,23,0.048673,0.036,-0.012673,0.522727,0.638889
4,texture,11,0,0,0.012168,0.000,-0.012168,0.000000,0.000000
10,structure_order,15,6,2,0.016593,0.006,-0.010593,0.133333,0.333333


## Шаг 4. Генерация клинического описания (Qwen2.5-7B)

In [ ]:
df = generate_descriptions_batch(df, device=device)
print("\nПример сгенерированного описания:")
print(df["description"].iloc[0])
# Сохранение результатов
output_csv = "features_with_descriptions.csv"
df.to_csv(output_csv, index=False)
print(f"\nСохранено: {len(df)} строк в {output_csv}")